# 一步步构建卷积神经网络

欢迎来到课程4的第一个作业！在此作业中，你将在 NumPy 中实现卷积 (CONV) 和池化 (POOL) 层，包括前向传播和（可选）反向传播。 

读完本 Notebook，你将能够： 

* 解释卷积运算
* 应用两种不同类型的池化操作
* 识别卷积神经网络中使用的组件（填充、步长、卷积核……）及其用途
* 构建卷积神经网络 

**符号**：
- 上标$[l]$表示$l^{th}$层的对象。 
    - 示例：$a^{[4]}$ 是 $4^{th}$ 层激活值。 $W^{[5]}$ 和 $b^{[5]}$ 是 $5^{th}$ 层参数。


- 上标 $(i)$ 表示 $i^{th}$ 示例中的对象。 
    - 示例：$x^{(i)}$ 是 $i^{th}$ 训练样本输入。
    
    
- 下标$i$表示向量的$i^{th}$条目。
    - 示例：$a^{[l]}_i$ 表示 $l$ 层中激活值的 $i^{th}$ 条目，假设这是一个全连接（FC）层。
    
    
- $n_H$、$n_W$ 和 $n_C$ 分别表示给定层的高度、宽度和通道数。如果要引用特定层$l$，也可以写为$n_H^{[l]}$、$n_W^{[l]}$、$n_C^{[l]}$。 
- $n_{H_{prev}}$、$n_{W_{prev}}$ 和 $n_{C_{prev}}$ 分别表示上一层的高度、宽度和通道数。如果引用特定层 $l$，则也可以表示为 $n_H^{[l-1]}$、$n_W^{[l-1]}$、$n_C^{[l-1]}$。 

你应该熟悉 `numpy` 和/或已经完成了该专项课程之前的课程。让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/DS4yP/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
- [2 - 作业概览](#2)
- [3 - 卷积神经网络](#3)
    - [3.1 - 零填充](#3-1)
        - [练习 1 - zero_pad](#ex-1)
    - [3.2 - 单步卷积](#3-2)
        - [练习 2 - conv_single_step](#ex-2)
    - [3.3 - 卷积神经网络 - 前向传播](#3-3)
        - [练习 3 - conv_forward](#ex-3)
- [4 - 池化层](#4)
    - [4.1 - 池化层的前向传播](#4-1)
        - [练习 4 - pool_forward](#ex-4)
- [5 - 反向传播中的卷积神经网络 (OPTIONAL / UNGRADED)](#5)
    - [5.1 - 卷积层反向传播](#5-1)
        - [5.1.1 - 计算 dA](#5-1-1)
        - [5.1.2 - 计算 dW](#5-1-2)
        - [5.1.3 - 计算数据库](#5-1-3)
            - [练习 5 - conv_backward](#ex-5)
    - [5.2 池化层 - 反向传播](#5-2)
        - [5.2.1 最大池化 - 反向传播](#5-2-1)
            - [练习 6 - create_mask_from_window](#ex-6)
        - [5.2.2 - 平均池化 - 反向传播](#5-2-2)
            - [练习 7 - distribute_value](#ex-7)
        - [5.2.3 整合池化层的反向传播](#5-2-3)
            - [练习 8 - pool_backward](#ex-8)

<a name='1'></a>
## 1 - 依赖库

首先导入你在此作业期间需要的所有包。 
- [NumPy](www.numpy.org) 是使用 Python 进行科学计算的基础包。
- [matplotlib](http://matplotlib.org) 是一个用 Python 绘制图形的库。
- np.random.seed(1) 用于保持所有随机函数调用的一致性。这有助于对你的工作进行评分。

In [ ]:
### v1.1

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # 设置绘图的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - 作业概览

你将实现卷积神经网络的构建块！你将实现的每个函数都会有详细的说明来引导你完成这些步骤：

- 卷积函数，包括：
    - 零填充
    - 卷积窗口 
    - 前向卷积
    - 卷积层的反向传播（选做）
- 池化函数，包括：
    - 池化层的前向传播
    - 创建蒙版 
    - 分配数值
    - 池化层的反向传播（选做）
    
这个Notebook会要求你在`numpy`中从头开始实现这些函数。在接下来的 Notebook 中，你将使用这些函数的 TensorFlow 等效项来构建以下模型：

<img src="images/model.png" style="width:800px;height:300px;">

**注意**：每个前向传播函数都有对应的反向传播函数。因此，前向传播的每一步都要缓存一些中间值，以便在反向传播时计算梯度。

<a name='3'></a>
## 3 - 卷积神经网络

尽管编程框架让卷积操作更容易使用，它仍是深度学习中较难理解的概念之一。如下图所示，卷积层将输入张量转换为形状不同的输出张量。

<img src="images/conv_nn.png" style="width:350px;height:200px;">

在这一部分中，你将构建卷积层的每一步。你将首先实现两个辅助函数：一个用于零填充，另一个用于计算卷积函数本身。 

<a name='3-1'></a>
### 3.1 - 零填充

零填充在图像边框周围添加零：

<img src="images/PAD.png" style="width:600px;height:400px;">
<div style="text-align:center;color:purple;"><u>  <b>图1</b> </u>： <b>Zero-Padding</b><br> 图像（3 通道，RGB），填充为 2。</div>

填充的主要好处是：

- 填充使 CONV 层能够在不减小输入张量高度和宽度的情况下进行卷积。这对构建深层网络很重要，否则高和宽会随网络加深而不断缩小。一个重要特例是 same 卷积，它使经过一层卷积后的高和宽保持不变。

- 它帮助我们在图像的边界保留更多信息。如果没有填充，下一层的值很少会受到图像边缘像素的影响。

<a name='ex-1'></a>
### 练习 1 - zero_pad
实现下面的函数，为一批样本 X 中的所有图像补零。可使用 [np.pad](https://docs.scipy.org/doc/numpy/reference/generated/numpy.pad.html)。例如，对于形状为 $(5,5,5,5,5)$ 的数组 a，若要在第 2 个维度设置 `pad = 1`，第 4 个维度设置 `pad = 3`，其余维度设置 `pad = 0`，可以这样写：
```python
a = np.pad(a, ((0,0), (1,1), (0,0), (3,3), (0,0)), mode='constant', constant_values = (0,0))
```

In [ ]:
# GRADED FUNCTION: zero_pad

def zero_pad(X, pad):
    """
    将数据集 X 的所有图像填充为零。填充应用于图像的高度和宽度， 
    ，如图 1 所示。
    
    论证：
    X -- python NumPy 形状为 (m, n_H, n_W, n_C) 的数组，表示一批 m 图像
    pad -- 整数，每个图像在垂直和水平尺寸上的填充量
    
    返回：
    X_pad -- 形状 (m, n_H + 2 * pad, n_W + 2 * pad, n_C) 的填充图像
    """
    
    #（≈ 1 行）
    # X_pad = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return X_pad

In [ ]:
np.random.seed(1)
x = np.random.randn(4, 3, 3, 2)
x_pad = zero_pad(x, 3)
print ("x.shape =\n", x.shape)
print ("x_pad.shape =\n", x_pad.shape)
print ("x[1,1] =\n", x[1, 1])
print ("x_pad[1,1] =\n", x_pad[1, 1])

fig, axarr = plt.subplots(1, 2)
axarr[0].set_title('x')
axarr[0].imshow(x[0, :, :, 0])
axarr[1].set_title('x_pad')
axarr[1].imshow(x_pad[0, :, :, 0])
zero_pad_test(zero_pad)

<a name='3-2'></a>
### 3.2 - 单步卷积 

在这部分中，实现单步卷积，其中将卷积核应用于输入的单个位置。这将用于构建一个卷积单元，其中： 

- 获取输入张量 
- 在输入的每个位置应用卷积核
- 输出另一个卷（通常大小不同）

<img src="images/Convolution_schematic.gif" style="width:500px;height:300px;">
<div style="text-align:center;color:purple;"><u>  <b>图2</b> </u>： <b>卷积运算</b><br>，卷积核为3x3，步长为1（步长=每次滑动时窗口移动的量）</div>

在计算机视觉应用中，左侧矩阵中的每个值对应于单个像素值。你可以将 3x3 卷积核与图像进行卷积，方法是将其值按元素与原始矩阵相乘，然后将它们相加并添加偏置。在练习的第一步中，你将实现单步卷积，对应于仅将卷积核应用于其中一个位置以获得单个实值输出。 

稍后在此 Notebook 中，你将将此函数应用于输入的多个位置以实现完整的卷积运算。 

<a name='ex-2'></a>
### 练习 2 - conv_single_step
实现`conv_single_step()`。 
    
[提示](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.sum.html).

**注意**：变量 b 将以 NumPy 数组的形式传入。  如果将标量（浮点或整数）添加到 NumPy 数组，则结果是 NumPy 数组。  在包含单个值的 NumPy 数组的特殊情况下，你可以将其转换为浮点数以将其转换为标量。

In [ ]:
# GRADED FUNCTION: conv_single_step

def conv_single_step(a_slice_prev, W, b):
    """
    在输出激活值的单个切片 (a_slice_prev) 上应用由参数 W 定义的一个卷积核 
    。
    
    参数：
    a_slice_prev -- 形状为 (f, f, n_C_prev) 的输入数据切片
    W -- 窗口中包含的权重参数 - 形状为 (f, f, n_C_prev) 的矩阵
    b -- 窗口中包含的偏置参数 - 形状为 (1, 1, 1) 的矩阵
    
    返回：
    Z -- 标量值，滑动窗口 (W, b) 在输入数据的切片 x 上进行卷积的结果
    """

    #（约3行代码）
    # a_slice_prev 和 W 之间的元素乘积。先不要添加偏置。
    # s = None
    # 卷 s 的所有条目的总和。
    # Z = None
    # 将偏置 b 添加到 Z。将 b 转换为 float()，以便 Z 产生标量值。
    # Z = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE

    return Z

In [ ]:
np.random.seed(1)
a_slice_prev = np.random.randn(4, 4, 3)
W = np.random.randn(4, 4, 3)
b = np.random.randn(1, 1, 1)

Z = conv_single_step(a_slice_prev, W, b)
print("Z =", Z)
conv_single_step_test(conv_single_step)

assert (type(Z) == np.float64), "You must cast the output to numpy float 64"
assert np.isclose(Z, -6.999089450680221), "Wrong value"

<a name='3-3'></a>
### 3.3 - 卷积神经网络 - 前向传播

在前向传播中，你将采用许多卷积核并将它们在输入上进行卷积。每个“卷积”都会为你提供一个二维矩阵输出。然后，你将堆叠这些输出以获得 3D 张量： 

<center>
<video width="620" height="440" src="images/conv_kiank.mp4" type="video/mp4" controls>
</video>
</center>

<a name='ex-3'></a>
### 练习 3 - conv_forward
实现下面的函数，将卷积核 `W` 与输入激活值 `A_prev` 进行卷积。  
这个函数采用以下输入：
* `A_prev`，前一层输出的激活值（对于一批m个输入）； 
* 权重用 `W` 表示。  卷积核窗口大小为 `f` 乘 `f`。
* 偏置向量为 `b`，每个卷积核对应一个独立的标量偏置。

你还可以访问超参数字典，其中包含步长和填充。 

**提示**： 
1. 要选择矩阵“a_prev”（形状 (5,5,3)）左上角的 2x2 切片，你可以执行以下操作：
```python
a_slice_prev = a_prev[0:2,0:2,:]
```
注意这如何给出高度为 2、宽度为 2、深度为 3 的 3D 切片。深度是通道数。  
当你使用你将定义的 `start/end` 索引来定义下面的 `a_slice_prev` 时，这将很有用。

2. 要定义 a_slice，你需要首先定义其角点 `vert_start`、`vert_end`、`horiz_start` 和 `horiz_end`。该图可能有助于你了解如何使用下面代码中的 h、w、f 和 s 定义每个角。

<img src="images/vert_horiz_kiank.png" style="width:400px;height:300px;">
<div style="text-align:center;color:purple;"><u>  <b>图 3</b> </u> : <b>定义使用垂直和水平开始/结束的切片（带有 2x2 卷积核）</b> <br> 该图仅显示单个通道。</div>


**提醒**：
    
将卷积的输出形状与输入形状相关的公式为：

$$
n_H = \Bigl\lfloor \frac{n_{H_{prev}} - f + 2 \times pad}{stride} \Bigr\rfloor +1
$$

$$
n_W = \Bigl\lfloor \frac{n_{W_{prev}} - f + 2 \times pad}{stride} \Bigr\rfloor +1
$$

$$
n_C = \text{number of filters used in the convolution}
$$

对于这个练习，不要担心向量化！只需使用 for 循环来实现所有内容即可。

#### 其他提示（如果你遇到困难）：


* 对以下变量使用数组切片（例如`varname[0:1,:,3:5]`）：  
  `a_prev_pad` ,`W`, `b`  
  - 复制函数的起始代码并在定义的函数之外的单独单元格中运行它。  
  - 检查每个数组的子集是否是你期望的大小和维度。  
* 要决定如何获取 `vert_start`、`vert_end`、`horiz_start`、`horiz_end`，请记住这些是前一层的索引。  
  - 绘制先前填充层（例如 8 x 8）和当前（输出层）（例如 2 x 2）的示例。  
  - 输出层的索引由 `h` 和 `w` 表示。  
* 确保 `a_slice_prev` 具有高度、宽度和深度。
* 请记住，`a_prev_pad` 是 `A_prev_pad` 的子集。  
  - 考虑一下 for 循环中应该使用哪一个。

In [ ]:
# GRADED FUNCTION: conv_forward

def conv_forward(A_prev, W, b, hparameters):
    """
    实现前向传播卷积函数
    
    参数：
    A_prev -- 输出上一层的激活值， 
        NumPy 形状数组 (m, n_H_prev, n_W_prev, n_C_prev)
    W -- 权重，NumPy 形状数组 (f, f, n_C_prev, n_C)
    b -- 偏置、NumPy 形状为 (1, 1, 1, n_C) 的数组
    hparameters -- 包含 "stride" 和 "pad" 的 python 字典
        
    返回：
    Z -- 转换输出，NumPy 形状数组 (m, n_H, n_W, n_C)
    cache -- conv_backward() 函数所需值的缓存
    """
    
    # 从 A_prev 的形状中检索尺寸（≈1 行代码）  
    # (m, n_H_prev, n_W_prev, n_C_prev) = 无
    
    # 从 W 的形状中检索尺寸（≈1 行代码）
    # (f, f, n_C_prev, n_C) = 无
    
    # 从 "hparameters" 检索信息（约 2 行）
    # stride = None
    # pad = None
    
    # 使用上面给出的公式计算 CONV 输出张量的尺寸。 
    # Hint：使用int()应用'floor'操作。 （约2行）
    # n_H = None
    # n_W = None
    
    # 将输出张量 Z 初始化为零。 （约1行）
    # Z = None
    
    # 通过填充 A_prev 创建 A_prev_pad
    # A_prev_pad = None
    
    # for i in range(None):               # 循环训练样本批次
        # a_prev_pad = None               # 选择训练样本的填充激活值
        # for h in range(None):           # 在输出张量的垂直轴上循环
            # 查找当前"slice"的垂直起点和终点（约2行）
            # vert_start = None
            # vert_end = None
            
            # for w in range(None):       # 在输出张量的水平轴上循环
                # 查找当前"slice"的水平起点和终点（约2行）
                # horiz_start = None
                # horiz_end = None
                
                # for c in range(None):   # 循环输出张量的通道（= #filters）
                                        
                    # 使用角定义 a_prev_pad 的 (3D) 切片（请参见单元格上方的提示）。 （约1行）
                    # a_slice_prev = None
                    
                    # 将三维切片与对应的卷积核 W 进行卷积并加上偏置 b，得到一个输出神经元的值。（约 3 行代码）
                    # weights = None
                    # biases = None
                    # Z[i, h, w, c] = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 将信息保存在 "cache" 中以用于反向传播
    cache = (A_prev, W, b, hparameters)
    
    return Z, cache

In [ ]:
np.random.seed(1)
A_prev = np.random.randn(2, 5, 7, 4)
W = np.random.randn(3, 3, 4, 8)
b = np.random.randn(1, 1, 1, 8)
hparameters = {"pad" : 1,
               "stride": 2}

Z, cache_conv = conv_forward(A_prev, W, b, hparameters)
z_mean = np.mean(Z)
z_0_2_1 = Z[0, 2, 1]
cache_0_1_2_3 = cache_conv[0][1][2][3]
print("Z's mean =\n", z_mean)
print("Z[0,2,1] =\n", z_0_2_1)
print("cache_conv[0][1][2][3] =\n", cache_0_1_2_3)

conv_forward_test_1(z_mean, z_0_2_1, cache_0_1_2_3)
conv_forward_test_2(conv_forward)


最后，CONV 层还应该包含激活值，在这种情况下，你将添加以下代码行：

```python
# 对窗口进行卷积以返回一个输出神经元
Z[i, h, w, c] = ...
# 申请激活值
A[i, h, w, c] = activation(Z[i, h, w, c])
```

不过，你不需要在这里执行此操作。 


<a name='4'></a>
## 4 - 池化层 

池化（POOL）层减少了输入的高度和宽度。它有助于减少计算量，并有助于使特征检测器对其在输入中的位置更加不变。两种类型的池化层是： 

- 最大池层：在输入上滑动（$f, f$）窗口，并将窗口的最大值存储在输出中。

- 平均池层：在输入上滑动（$f, f$）窗口，并将窗口的平均值存储在输出中。

<table>
<tr>
<td><img src="images/max_pool1.png" style="width:500px;height:300px;"></td>
<td><img src="images/a_pool.png" style="width:500px;height:300px;"></td>
</tr>
</table>

这些池化层没有可供反向传播训练的参数。但是，它们具有超参数，例如窗口大小 $f$。这指定了你要计算*最大值*或*平均值*的 $f \times f$ 窗口的高度和宽度。 

<a name='4-1'></a>
### 4.1 - 池化层的前向传播
现在，你将在同一个函数中实现 MAX-POOL 和 AVG-POOL。 

<a name='ex-4'></a>
### 练习 4 - pool_forward

实现池化层的前向传播。请按照下面评论中的提示进行操作。

**提醒**：
由于没有填充，将池化的输出形状绑定到输入形状的公式为：

$$n_H = \Bigl\lfloor \frac{n_{H_{prev}} - f}{stride} \Bigr\rfloor +1$$

$$n_W = \Bigl\lfloor \frac{n_{W_{prev}} - f}{stride} \Bigr\rfloor +1$$

$$n_C = n_{C_{prev}}$$




In [ ]:
# GRADED FUNCTION: pool_forward

def pool_forward(A_prev, hparameters, mode = "max"):
    """
    实现池化层的前向传播
    
    参数：
    A_prev -- 输入数据，NumPy 形状数组 (m, n_H_prev, n_W_prev, n_C_prev)
    hparameters -- 包含 "f" 和 "stride" 的 python 字典
    mode -- 你要使用的池模式，定义为字符串（"max" 或 "average"）
    
    返回：
    A -- 池化层的输出，形状为 (m, n_H, n_W, n_C) 的 NumPy 数组
    cache -- 池化层反向传播中使用的缓存，包含输入和h参数 
    """
    
    # 从输入形状中检索尺寸
    (m, n_H_prev, n_W_prev, n_C_prev) = A_prev.shape
    
    # 从"hparameters"中检索超参数
    f = hparameters["f"]
    stride = hparameters["stride"]
    
    # 定义输出的尺寸
    n_H = int(1 + (n_H_prev - f) / stride)
    n_W = int(1 + (n_W_prev - f) / stride)
    n_C = n_C_prev
    
    # 初始化输出矩阵A
    A = np.zeros((m, n_H, n_W, n_C))              
    
    # for i in range(None):                         # 循环训练样本
        # for h in range(None):                     # 输出张量纵轴循环
            # 查找当前"slice"的垂直起点和终点（约2行）
            # vert_start = None
            # vert_end = None
            
            # for w in range(None):                 # 输出张量横轴循环
                # 查找当前"slice"的垂直起点和终点（约2行）
                # horiz_start = None
                # horiz_end = None
                
                # for c in range (None):            # 循环输出张量的通道
                    
                    # 使用角点定义 A_prev 的第 i 个训练样本上的当前切片，通道 c。 （约1行）
                    # a_prev_slice = None
                    
                    # 计算切片上的池化操作。 
                    # 使用 if 语句来区分模式。 
                    # 使用np.max和np.mean。
                    # if mode == "max":
                        # A[i, h, w, c] = None
                    # elif mode == "average":
                        # A[i, h, w, c] = None
    
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 将 pool_backward() 的输入和 h 参数存储在 "cache" 中
    cache = (A_prev, hparameters)
    
    # 确保输出形状正确
    #assert(A.shape == (m, n_H, n_W, n_C))
    
    return A, cache

In [ ]:
# 情况 1：步长为 1
print("CASE 1:\n")
np.random.seed(1)
A_prev_case_1 = np.random.randn(2, 5, 5, 3)
hparameters_case_1 = {"stride" : 1, "f": 3}

A, cache = pool_forward(A_prev_case_1, hparameters_case_1, mode = "max")
print("mode = max")
print("A.shape = " + str(A.shape))
print("A[1, 1] =\n", A[1, 1])
A, cache = pool_forward(A_prev_case_1, hparameters_case_1, mode = "average")
print("mode = average")
print("A.shape = " + str(A.shape))
print("A[1, 1] =\n", A[1, 1])

pool_forward_test_1(pool_forward)

# 情况 2：步长为 2
print("\n\033[0mCASE 2:\n")
np.random.seed(1)
A_prev_case_2 = np.random.randn(2, 5, 5, 3)
hparameters_case_2 = {"stride" : 2, "f": 3}

A, cache = pool_forward(A_prev_case_2, hparameters_case_2, mode = "max")
print("mode = max")
print("A.shape = " + str(A.shape))
print("A[0] =\n", A[0])
print()

A, cache = pool_forward(A_prev_case_2, hparameters_case_2, mode = "average")
print("mode = average")
print("A.shape = " + str(A.shape))
print("A[1] =\n", A[1])

pool_forward_test_2(pool_forward)

**预期输出**

```
CASE 1:

mode = max
A.shape = (2, 3, 3, 3)
A[1, 1] =
 [[1.96710175 0.84616065 1.27375593]
 [1.96710175 0.84616065 1.23616403]
 [1.62765075 1.12141771 1.2245077 ]]
mode = average
A.shape = (2, 3, 3, 3)
A[1, 1] =
 [[ 0.44497696 -0.00261695 -0.31040307]
 [ 0.50811474 -0.23493734 -0.23961183]
 [ 0.11872677  0.17255229 -0.22112197]]
All tests passed!

CASE 2:

mode = max
A.shape = (2, 2, 2, 3)
A[0] =
 [[[1.74481176 0.90159072 1.65980218]
  [1.74481176 1.6924546  1.65980218]]

 [[1.13162939 1.51981682 2.18557541]
  [1.13162939 1.6924546  2.18557541]]]

mode = average
A.shape = (2, 2, 2, 3)
A[1] =
 [[[-0.17313416  0.32377198 -0.34317572]
  [ 0.02030094  0.14141479 -0.01231585]]

 [[ 0.42944926  0.08446996 -0.27290905]
  [ 0.15077452  0.28911175  0.00123239]]]
All tests passed!
```

<div style="color:blue;">

**你应该记住什么**：

* 卷积通过获取输入数据和 3D 权重数组（卷积核）之间的点积来从输入图像中提取特征。 
* 卷积的2D输出称为特征图
* 卷积层是卷积核在图像上滑动并计算点积的地方 
    * 这会将输入张量转换为不同大小的输出张量 
* 零填充有助于在图像边界保留更多信息，并且有助于构建更深的网络，因为你可以在不缩小张量的高度和宽度的情况下构建 CONV 层
* 池化层通过在每个指定区域上滑动 2D 窗口来逐渐减小输入的高度和宽度，然后汇总该区域中的特征

</div>

**恭喜**！你现在已经实现了卷积网络所有层的前向传播。干得好！

Notebook 的其余部分是可选的，不会评分。如果你继续，请记住先点击“提交”按钮提交你的作业进行评分。 

<a name='5'></a>
## 5 - 卷积神经网络中的反向传播（可选/不计分）

在现代深度学习框架中，你只需要实现前向传播，框架就会照顾反向传播，所以大多数深度学习工程师不需要关心反向传播的细节。卷积网络的反向传播很复杂。如果你愿意，你可以通过 Notebook 的这个可选部分来了解卷积网络中的反向传播是什么样的。 

在此前实现全连接神经网络时，你已经使用反向传播计算代价相对于参数的导数，并据此更新参数。卷积神经网络也遵循同样的原则。相关反向传播公式较为复杂，课堂上未作推导，下面将简要列出。

<a name='5-1'></a>
### 5.1 - 卷积层反向传播 

让我们从为 CONV 层实现反向传播开始。 

<a name='5-1-1'></a>
#### 5.1.1 - 计算 dA：
下面的公式计算给定训练样本和卷积核 $W_c$ 对代价相对于输入的梯度 $dA$ 的贡献：

$$dA \mathrel{+}= \sum _{h=0} ^{n_H} \sum_{w=0} ^{n_W} W_c \times dZ_{hw} \tag{1}$$

其中，$W_c$ 是卷积核，$dZ_{hw}$ 是代价相对于卷积层输出 Z 的第 h 行、第 w 列元素的梯度，是一个标量。更新 dA 时，将同一个卷积核 $W_c$ 乘以各位置对应的 dZ，并累加到相应输入切片上。这是因为前向传播时，同一个卷积核会与不同的 a_slice 做逐元素乘法并求和；反向传播时，需要累加所有相关切片对输入梯度的贡献。

在代码中，在适当的 for 循环内，此公式转换为：
```python
da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += W[:,:,:,c] * dZ[i, h, w, c]
```

<a name='5-1-2'></a>
#### 5.1.2 - 计算 dW：
下面的公式计算损失相对于卷积核的梯度 $dW_c$（$dW_c$ 对应一个卷积核的梯度）：

$$dW_c  \mathrel{+}= \sum _{h=0} ^{n_H} \sum_{w=0} ^ {n_W} a_{slice} \times dZ_{hw}  \tag{2}$$

其中 $a_{slice}$ 对应于用于生成激活值 $Z_{ij}$ 的切片。因此，这最终为我们提供了相对于该切片的 $W$ 的梯度。由于它是相同的 $W$，因此我们只需将所有此类梯度相加即可得到 $dW$。 

在代码中，在适当的 for 循环内，此公式转换为：
```python
dW[:,:,:,c] += a_slice * dZ[i, h, w, c]
```

<a name='5-1-3'></a>
#### 5.1.3 - 计算 db：

下面的公式计算代价相对于卷积核 $W_c$ 所对应偏置的梯度 $db$：

$$db = \sum_h \sum_w dZ_{hw} \tag{3}$$

正如你之前在基本神经网络中所见，db 是通过对 $dZ$ 求和来计算的。在这种情况下，你只需根据代价对转换输出 (Z) 的所有梯度进行求和。 

在代码中，在适当的 for 循环内，此公式转换为：
```python
db[:,:,:,c] += dZ[i, h, w, c]
```

<a name='ex-5'></a>
### 练习 5 - conv_backward

实现下面的 `conv_backward` 函数。你应该对所有训练样本、卷积核、高度和宽度求和。然后，你应该使用上面的公式 1、2 和 3 计算导数。 

In [ ]:
def conv_backward(dZ, cache):
    """
    为卷积实现反向传播函数
    
    参数：
    dZ -- 代价相对于卷积层输出 Z 的梯度，为形状 (m, n_H, n_W, n_C) 的 NumPy 数组
    cache -- conv_backward() 所需值的缓存，conv_forward() 的输出
    
    返回：
    dA_prev -- 代价相对于卷积层输入 A_prev 的梯度，
               NumPy 形状数组 (m, n_H_prev, n_W_prev, n_C_prev)
    dW -- 代价相对于卷积层权重 W 的梯度
          NumPy 形状数组 (f, f, n_C_prev, n_C)
    db -- 代价相对于卷积层偏置 b 的梯度
          NumPy 形状数组 (1, 1, 1, n_C)
    """    
    
        
    # 从 "cache" 检索信息
    # (A_prev, W, b, hparameters) = 无
    # 从 A_prev 的形状中检索尺寸
    # (m, n_H_prev, n_W_prev, n_C_prev) = 无
    # 从 W 形状检索尺寸
    # (f, f, n_C_prev, n_C) = 无
    
    # 从 "hparameters" 检索信息
    # stride = None
    # pad = None
    
    # 从 dZ 的形状中检索尺寸
    # (m, n_H, n_W, n_C) = 无
    
    # 使用正确的形状初始化 dA_prev、dW、db
    # dA_prev = None                          
    # dW = None
    # db = None
    
    # 焊盘 A_prev 和 dA_prev
    # A_prev_pad = zero_pad(A_prev, pad)
    # dA_prev_pad = zero_pad(dA_prev, pad)
    
    #for i in range(m):                       # 循环训练样本
        
        # 从 A_prev_pad 和 dA_prev_pad 中选择训练样本
        # a_prev_pad = None
        # da_prev_pad = None
        
        #for h in range(n_H):                   # 在输出张量的垂直轴上循环
        #    for w in range(n_W):               # 在输出张量的水平轴上循环
        #        for c in range(n_C):           # 循环输出张量的通道
                    
                    # 找到当前"slice"的角点
                    # vert_start = None
                    # vert_end = None
                    # horiz_start = None
                    # horiz_end = None

                    # 使用角点定义 a_prev_pad 的切片
                    # a_slice = None

                    # 使用上面给出的代码公式更新窗口和卷积核参数的梯度
                    # da_prev_pad[vert_start:vert_end, horiz_start:horiz_end, :] += None
                    # dW[:,:,:,c] += None
                    # db[:,:,:,c] += None
                    
        # 将第 i 个训练样本的 dA_prev 设置为未填充的 da_prev_pad （提示：使用 X[pad:-pad, pad:-pad, :]）
        # dA_prev[i, :, :, :] = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 确保输出形状正确
    assert(dA_prev.shape == (m, n_H_prev, n_W_prev, n_C_prev))
    
    return dA_prev, dW, db

In [ ]:
# 我们将运行 conv_forward 来初始化 'Z' 和 'cache_conv"，
# 我们将用它来测试 conv_backward 函数
np.random.seed(1)
A_prev = np.random.randn(10, 4, 4, 3)
W = np.random.randn(2, 2, 3, 8)
b = np.random.randn(1, 1, 1, 8)
hparameters = {"pad" : 2,
               "stride": 2}
Z, cache_conv = conv_forward(A_prev, W, b, hparameters)

# 测试conv_backward
dA, dW, db = conv_backward(Z, cache_conv)

print("dA_mean =", np.mean(dA))
print("dW_mean =", np.mean(dW))
print("db_mean =", np.mean(db))

assert type(dA) == np.ndarray, "Output must be a np.ndarray"
assert type(dW) == np.ndarray, "Output must be a np.ndarray"
assert type(db) == np.ndarray, "Output must be a np.ndarray"
assert dA.shape == (10, 4, 4, 3), f"Wrong shape for dA  {dA.shape} != (10, 4, 4, 3)"
assert dW.shape == (2, 2, 3, 8), f"Wrong shape for dW {dW.shape} != (2, 2, 3, 8)"
assert db.shape == (1, 1, 1, 8), f"Wrong shape for db {db.shape} != (1, 1, 1, 8)"
assert np.isclose(np.mean(dA), 1.4524377), "Wrong values for dA"
assert np.isclose(np.mean(dW), 1.7269914), "Wrong values for dW"
assert np.isclose(np.mean(db), 7.8392325), "Wrong values for db"

print("\033[92m All tests passed.")

**预期输出**：
<table>
    <tr>
        <td>
            dA_mean
        </td>
        <td>
            1.45243777754
        </td>
    </tr>
    <tr>
        <td>
            dW_mean
        </td>
        <td>
            1.72699145831
        </td>
    </tr>
    <tr>
        <td>
            db_mean
        </td>
        <td>
            7.83923256462
        </td>
    </tr>

</table>


<a name='5-2'></a>
## 5.2 池化层 - 反向传播

接下来，让我们从 MAX-POOL 层开始，实现池化层的反向传播。即使池化层没有用于更新反向传播的参数，你仍然需要通过池化层反向传播梯度，以便为池化层之前的层计算梯度。 

<a name='5-2-1'></a>
### 5.2.1 最大池化 - 反向传播  

在进入池化层的反向传播之前，你将构建一个名为 `create_mask_from_window()` 的辅助函数，它执行以下操作： 

$$ X = \begin{bmatrix}
1 && 3 \\
4 && 2
\end{bmatrix} \quad \rightarrow  \quad M =\begin{bmatrix}
0 && 0 \\
1 && 0
\end{bmatrix}\tag{4}$$

正如你所看到的，这个函数创建了一个“掩码”矩阵，用于跟踪矩阵最大值的位置。 True (1) 表示 X 中最大值的位置，其他条目为 False (0)。稍后你将看到平均池化的反向传播与此类似，但使用不同的掩码。  

<a name='ex-6'></a>
### 练习 6 - create_mask_from_window

实现`create_mask_from_window()`。这个函数将有助于向后池化。 
提示：
- [np.max()]() 可能会有所帮助。它计算数组的最大值。
- 如果你有一个矩阵 X 和一个标量 x：`A = (X == x)` 将返回一个与 X 大小相同的矩阵 A，这样：
```
A[i,j] = True if X[i,j] = x
A[i,j] = False if X[i,j] != x
```
 - 在这里，你不需要考虑矩阵中有多个最大值的情况。

In [ ]:
def create_mask_from_window(x):
    """
    从输入矩阵 x 创建掩码，以识别 x 的最大条目。
    
    参数：
    x -- 形状为 (f, f) 的数组
    
    返回：
    mask -- 与 window 形状相同的数组，在与 x 的最大条目相对应的位置包含 True。
    """    
    # （约1行）
    # mask = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return mask

In [ ]:
np.random.seed(1)
x = np.random.randn(2, 3)
mask = create_mask_from_window(x)
print('x = ', x)
print("mask = ", mask)

x = np.array([[-1, 2, 3],
              [2, -3, 2],
              [1, 5, -2]])

y = np.array([[False, False, False],
     [False, False, False],
     [False, True, False]])
mask = create_mask_from_window(x)

assert type(mask) == np.ndarray, "Output must be a np.ndarray"
assert mask.shape == x.shape, "Input and output shapes must match"
assert np.allclose(mask, y), "Wrong output. The True value must be at position (2, 1)"

print("\033[92m All tests passed.")

**预期输出：** 

<table> 
<tr> 
<td>

<strong>x =</strong>
</td>

<td>

[[ 1.62434536 -0.61175641 -0.52817175] <br>
 [-1.07296862  0.86540763 -2.3015387 ]]

  </td>
</tr>

<tr> 
<td>
掩码=
</td>
<td>
[[真假假]<br>
 【假假假】]
</td>
</tr>


</table>

为什么要跟踪最大值的位置？这是因为这是最终影响产出的输入值，从而影响代价。 Backprop 正在根据代价计算梯度，因此影响最终代价的任何因素都应该具有非零梯度。因此，反向传播会将梯度“传播”回影响代价的特定输入值。 

<a name='5-2-2'></a>
### 5.2.2 - 平均池化 - 反向传播 

在最大池化中，对于每个输入窗口，对输出的所有“影响”都来自单个输入值——最大值。在平均池化中，输入窗口的每个元素对输出都有相同的影响。因此，为了实现反向传播，你现在将实现一个反映这一点的帮助器函数。

例如，若前向传播使用 2x2 窗口进行平均池化，则反向传播使用的掩码如下：
$$ dZ = 1 \quad \rightarrow  \quad dZ =\begin{bmatrix}
1/4 && 1/4 \\
1/4 && 1/4
\end{bmatrix}\tag{5}$$

这意味着 $dZ$ 矩阵中的每个位置对输出的贡献相等，因为在前向传播中，我们取了平均值。 

<a name='ex-7'></a>
### 练习 7 - distribute_value

实现下面的函数，通过维度形状矩阵平均分配值dz。 

[提示](https://docs.scipy.org/doc/numpy-1.13.0/reference/generated/numpy.ones.html)

In [ ]:
def distribute_value(dz, shape):
    """
    将输入值分布在维度形状矩阵中
    
    参数：
    dz -- 输入标量
    shape -- 我们要为其分配 dz 值的输出矩阵的形状 (n_H, n_W)
    
    返回：
    a -- 大小为 (n_H, n_W) 的数组，我们为其分配 dz 值
    """    
    # 从形状中检索尺寸（≈1 行代码）
    # (n_H, n_W) = 无
    
    # 计算要在矩阵上分布的值（≈1 行）
    # average = None
    
    # 创建一个矩阵，其中每个条目都是 "average" 值（约 1 行）
    # a = None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return a

In [ ]:
a = distribute_value(2, (2, 2))
print('distributed value =', a)


assert type(a) == np.ndarray, "Output must be a np.ndarray"
assert a.shape == (2, 2), f"Wrong shape {a.shape} != (2, 2)"
assert np.sum(a) == 2, "Values must sum to 2"

a = distribute_value(100, (10, 10))
assert type(a) == np.ndarray, "Output must be a np.ndarray"
assert a.shape == (10, 10), f"Wrong shape {a.shape} != (10, 10)"
assert np.sum(a) == 100, "Values must sum to 100"

print("\033[92m All tests passed.")

**预期输出**： 

<table> 
<tr> 
<td>
distributed_value =
</td>
<td>
[[ 0.5  0.5]
<br> 
[ 0.5  0.5]]
</td>
</tr>
</table>

<a name='5-2-3'></a>
### 5.2.3 整合池化层的反向传播 

你现在拥有在池化层上计算反向传播所需的一切。

<a name='ex-8'></a>
### 练习 8 - pool_backward

在两种模式（`"max"` 和 `"average"`）下实现 `pool_backward` 函数。你将再次使用 4 个 for 循环（迭代训练样本、高度、宽度和通道）。你应该使用 `if/elif` 语句来查看模式是否等于 `'max'` 或 `'average'`。如果它等于“平均”，你应该使用上面实现的 `distribute_value()` 函数来创建与 `a_slice` 形状相同的矩阵。否则，模式等于“`max`”，你将使用 `create_mask_from_window()` 创建一个掩码，并将其乘以相应的 dA 值。

In [ ]:
def pool_backward(dA, cache, mode = "max"):
    """
    实现池化层的反向传播
    
    参数：
    dA -- 代价相对于池化层输出的梯度，形状与 A 相同
    cache -- 缓存池化层前向传播的输出，包含该层的输入和h参数 
    mode -- 你要使用的池模式，定义为字符串（"max" 或 "average"）
    
    返回：
    dA_prev -- 代价相对于池化层输入的梯度，形状与 A_prev 相同
    """
    # 从缓存中检索信息（约1行）
    # (A_prev, hparameters) = 无
    
    # 从 "hparameters" 检索超参数（约 2 行）
    # stride = None
    # f = None
    
    # 从 A_prev 的形状和 dA 的形状中检索尺寸（约 2 行代码）
    # m, n_H_prev, n_W_prev, n_C_prev = None
    # m, n_H, n_W, n_C = None
    
    # 用零初始化 dA_prev（约 1 行）
    # dA_prev = None
    
    # for i in range(None): # 循环训练样本
        
        # 从A_prev中选择训练样本（约1行）
        # a_prev = None
        
        # for h in range(n_H):                   # 垂直轴上的循环
            # for w in range(n_W):               # 水平轴上的循环
                # for c in range(n_C):           # 循环通道（深度）
        
                    # 找到当前"slice"的角点（≈4行）
                    # vert_start = None
                    # vert_end = None
                    # horiz_start = None
                    # horiz_end = None
                    
                    # 在两种模式下计算反向传播。
                    # if mode == "max":
                        
                        # 使用角点和 "c" 定义 a_prev 中的当前切片（约 1 行）
                        # a_prev_slice = None
                        
                        # 从 a_prev_slice 创建蒙版（约 1 行）
                        # mask = None

                        # 将 dA_prev 设置为 dA_prev +（掩码乘以 dA 的正确输入）（≈1 行）
                        # dA_prev[i, vert_start: vert_end, horiz_start: horiz_end, c] += None
                        
                    # elif mode == "average":
                        
                        # 从 dA 获取值 da（≈1 行）
                        # da = None
                        
                        # 将池化窗口的形状定义为 fxf（约 1 行代码）
                        # shape = None

                        # 分发它以获得正确的 dA_prev 切片。 即添加da的分布值。 （约1行）
                        # dA_prev[i, vert_start: vert_end, horiz_start: horiz_end, c] += None
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    # 确保输出形状正确
    assert(dA_prev.shape == A_prev.shape)
    
    return dA_prev

In [ ]:
np.random.seed(1)
A_prev = np.random.randn(5, 5, 3, 2)
hparameters = {"stride" : 1, "f": 2}
A, cache = pool_forward(A_prev, hparameters)
print(A.shape)
print(cache[0].shape)
dA = np.random.randn(5, 4, 2, 2)

dA_prev1 = pool_backward(dA, cache, mode = "max")
print("mode = max")
print('mean of dA = ', np.mean(dA))
print('dA_prev1[1,1] = ', dA_prev1[1, 1])  
print()
dA_prev2 = pool_backward(dA, cache, mode = "average")
print("mode = average")
print('mean of dA = ', np.mean(dA))
print('dA_prev2[1,1] = ', dA_prev2[1, 1]) 

assert type(dA_prev1) == np.ndarray, "Wrong type"
assert dA_prev1.shape == (5, 5, 3, 2), f"Wrong shape {dA_prev1.shape} != (5, 5, 3, 2)"
assert np.allclose(dA_prev1[1, 1], [[0, 0], 
                                    [ 5.05844394, -1.68282702],
                                    [ 0, 0]]), "Wrong values for mode max"
assert np.allclose(dA_prev2[1, 1], [[0.08485462,  0.2787552], 
                                    [1.26461098, -0.25749373], 
                                    [1.17975636, -0.53624893]]), "Wrong values for mode average"
print("\033[92m All tests passed.")

**预期输出**： 

模式=最大：
<table> 
<tr> 
<td>

<strong>dA 的平均值 =</strong>
</td>

<td>

0.145713902729

  </td>
</tr>

<tr> 
<td>
dA_prev[1,1] =
</td>
<td>
[[ 0.          0.        ] <br>
 [ 5.05844394 -1.68282702] <br>
 [ 0.          0.        ]]
</td>
</tr>
</table>

模式=平均
<table> 
<tr> 
<td>

dA 的平均值 =
</td>

<td>

0.145713902729

  </td>
</tr>

<tr> 
<td>
dA_prev[1,1] =
</td>
<td>
[[ 0.08485462  0.2787552 ] <br>
 [ 1.26461098 -0.25749373] <br>
 [ 1.17975636 -0.53624893]]
</td>
</tr>
</table>

**恭喜**！你已完成作业及其可选部分。你现在了解了卷积神经网络的工作原理，并实现了神经网络的所有构建模块。在下一个作业中，你将使用 TensorFlow 实现 ConvNet。干得好！那里见。